# Flower Classification - Complete Pipeline
## End-to-End Machine Learning Workflow

This master notebook orchestrates the complete flower classification pipeline from EDA to model evaluation.

### Pipeline Steps:
1. **EDA** - Exploratory Data Analysis
2. **Data Preprocessing** - Loading, cleaning, scaling
3. **Feature Engineering** - Creating new features
4. **Model Training** - Training multiple classifiers
5. **Model Evaluation** - Comprehensive evaluation

## Setup and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print('✅ Libraries loaded successfully')

## Quick Start - Run Entire Pipeline

In [ ]:
# Run this cell to execute the complete pipeline
from IPython.display import display, Markdown

display(Markdown('### 🚀 Starting Complete Pipeline...'))

# Pipeline configuration
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

print(f'Configuration:')
print(f'  - Random State: {RANDOM_STATE}')
print(f'  - Test Size: {TEST_SIZE}')
print(f'  - CV Folds: {CV_FOLDS}')

## Step 1: Load and Explore Data

In [ ]:
# Load data from CSV file
df = pd.read_csv('data/raw/iris.csv')

# Rename columns
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

display(Markdown(f'**Dataset Shape:** {df.shape}'))
display(Markdown(f'**Features:** sepal_length, sepal_width, petal_length, petal_width'))
display(Markdown(f'**Classes:** {list(df["species"].unique())}'))
display(Markdown(f'**Class Distribution:**\n{df["species"].value_counts().to_string()}'))

df.head()

In [ ]:
# Quick EDA visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
features = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

for i, feature in enumerate(features):
    ax = axes[i // 2, i % 2]
    sns.boxplot(data=df, x='species_name', y=feature, ax=ax, palette='Set2')
    ax.set_title(feature)
    ax.set_xlabel('')

plt.suptitle('Feature Distribution by Species', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 2: Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Separate features and target
X = df[features]
y = df['species']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Train size: {len(X_train)}')
print(f'Test size: {len(X_test)}')

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Encode target
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print('✅ Data preprocessing complete')

## Step 3: Feature Engineering

In [ ]:
# Create ratio features
df_engineered = X.copy()
df_engineered['sepal_ratio'] = df_engineered['sepal length (cm)'] / (df_engineered['sepal width (cm)'] + 1e-6)
df_engineered['petal_ratio'] = df_engineered['petal length (cm)'] / (df_engineered['petal width (cm)'] + 1e-6)

# Create area features
df_engineered['sepal_area'] = df_engineered['sepal length (cm)'] * df_engineered['sepal width (cm)']
df_engineered['petal_area'] = df_engineered['petal length (cm)'] * df_engineered['petal width (cm)']

print(f'Original features: {len(features)}')
print(f'Engineered features: {len(df_engineered.columns)}')
print('✅ Feature engineering complete')

df_engineered.head()

## Step 4: Model Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, multi_class='multinomial'),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'XGBoost': xgb.XGBClassifier(random_state=RANDOM_STATE, n_estimators=100, eval_metric='mlogloss', use_label_encoder=False)
}

# Train models
trained_models = {}
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train_encoded)
    trained_models[name] = model
    
    train_acc = model.score(X_train_scaled, y_train_encoded)
    test_acc = model.score(X_test_scaled, y_test_encoded)
    
    results.append({
        'Model': name,
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc
    })
    print(f'{name}: Train={train_acc:.4f}, Test={test_acc:.4f}')

print('✅ Model training complete')

In [ ]:
# Results visualization
results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)

plt.figure(figsize=(10, 6))
bars = plt.bar(results_df['Model'], results_df['Test Accuracy'], color=sns.color_palette('viridis', len(results_df)))
plt.xticks(rotation=45, ha='right')
plt.ylabel('Accuracy')
plt.title('Model Comparison - Test Accuracy')
plt.ylim(0, 1.05)

for bar, val in zip(bars, results_df['Test Accuracy']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]['Model']
print(f'\n🏆 Best Model: {best_model_name} (Accuracy: {results_df.iloc[0]["Test Accuracy"]:.4f})')

## Step 5: Model Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Get best model predictions
best_model = trained_models[best_model_name]
y_pred = best_model.predict(X_test_scaled)

# Confusion Matrix
cm = confusion_matrix(y_test_encoded, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.tight_layout()
plt.show()

# Classification Report
print(f'\nClassification Report - {best_model_name}')
print('='*60)
print(classification_report(y_test_encoded, y_pred, target_names=label_encoder.classes_, zero_division=0))

In [ ]:
# Calculate additional metrics
from sklearn.metrics import precision_score, recall_score, f1_score

metrics = {
    'Accuracy': accuracy_score(y_test_encoded, y_pred),
    'Precision': precision_score(y_test_encoded, y_pred, average='weighted', zero_division=0),
    'Recall': recall_score(y_test_encoded, y_pred, average='weighted', zero_division=0),
    'F1 Score': f1_score(y_test_encoded, y_pred, average='weighted', zero_division=0)
}

print('Final Metrics:')
print('='*60)
for metric, value in metrics.items():
    print(f'{metric}: {value:.4f}')

# Visualize metrics
plt.figure(figsize=(10, 5))
bars = plt.bar(metrics.keys(), metrics.values(), color=sns.color_palette('viridis', len(metrics)))
plt.ylabel('Score')
plt.title(f'Performance Metrics - {best_model_name}')
plt.ylim(0, 1.05)

for bar, val in zip(bars, metrics.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Step 6: Save Artifacts

In [ ]:
import joblib

# Create directories
models_dir = Path('../models/artifacts')
data_dir = Path('../data/processed')
models_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

# Save best model
joblib.dump(best_model, models_dir / 'best_model.joblib')
joblib.dump(scaler, models_dir / 'scaler.joblib')
joblib.dump(label_encoder, models_dir / 'label_encoder.joblib')
joblib.dump(features, models_dir / 'feature_columns.joblib')

# Save preprocessed data
np.save(data_dir / 'X_train.npy', X_train_scaled)
np.save(data_dir / 'X_test.npy', X_test_scaled)
np.save(data_dir / 'y_train.npy', y_train_encoded)
np.save(data_dir / 'y_test.npy', y_test_encoded)

# Save results
results_df.to_csv(models_dir / 'pipeline_results.csv', index=False)

print('✅ Saved artifacts:')
print('  - best_model.joblib')
print('  - scaler.joblib')
print('  - label_encoder.joblib')
print('  - feature_columns.joblib')
print('  - Preprocessed data (X_train, X_test, y_train, y_test)')
print('  - pipeline_results.csv')

## Pipeline Summary

In [ ]:
from IPython.display import display, Markdown

display(Markdown('''
## 🎉 Pipeline Complete!

### Summary:
- **Dataset**: Iris Flower Dataset (150 samples, 4 features, 3 classes)
- **Preprocessing**: StandardScaler normalization, LabelEncoder encoding
- **Feature Engineering**: Added ratio and area features
- **Models Trained**: Logistic Regression, Random Forest, Decision Tree, XGBoost
- **Best Model**: See results above
- **Final Accuracy**: See metrics above

### Next Steps:
1. Run individual notebooks for detailed analysis
2. Use the saved model for predictions
3. Deploy the model using the FastAPI backend
'''))